In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# 1. LOAD DATASET

In [ ]:
FILE_PATH = 'data/online_retail.csv'
df = pd.read_csv(FILE_PATH, encoding='ISO-8859-1')
print('Original shape:', df.shape)
print(df.head())
print(df.info())
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
# 2. CLEAN DATA

In [ ]:
df.columns = df.columns.str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')

# Remove records without customer ID
# Remove cancelled invoices
# Remove invalid quantity/price and invalid dates
df = df.dropna(subset=['CustomerID', 'InvoiceDate'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

print('\nCleaned shape:', df.shape)

In [ ]:
# 3. DESCRIPTIVE STATISTICS

In [ ]:
avg_purchase_value = df['TotalAmount'].mean()
total_orders = df['InvoiceNo'].nunique()
total_customers = df['CustomerID'].nunique()
avg_order_value = df.groupby('InvoiceNo')['TotalAmount'].sum().mean()

customer_sales = df.groupby('CustomerID')['TotalAmount'].sum()
customer_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()

# Customer Lifetime Value proxy: total historical spend per customer
customer_ltv = customer_sales

print('\n--- Descriptive Statistics ---')
print(f'Average transaction purchase value: £{avg_purchase_value:,.2f}')
print(f'Average order value: £{avg_order_value:,.2f}')
print(f'Purchase frequency (average orders/customer): {customer_orders.mean():.2f}')
print(f'Average customer lifetime value (historical spend): £{customer_ltv.mean():,.2f}')
print(f'Total customers: {total_customers:,}')
print(f'Total orders: {total_orders:,}')

In [ ]:
# 4. EDA

In [ ]:
monthly_revenue = df.set_index('InvoiceDate').resample('ME')['TotalAmount'].sum()
plt.figure(figsize=(11, 5))
plt.plot(monthly_revenue.index, monthly_revenue.values, marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('output/monthly_revenue.png', dpi=150)
plt.show()

In [ ]:
# 5. RFM FEATURES

In [ ]:
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalAmount', 'sum')
).reset_index()

print('\nRFM sample:')
print(rfm.head())

# Customer lifetime value is represented by Monetary in this transaction-history dataset.
rfm['CustomerLifetimeValue'] = rfm['Monetary']

In [ ]:
# 6. HANDLE OUTLIERS

In [ ]:
model_rfm = rfm.copy()
for col in ['Recency', 'Frequency', 'Monetary']:
    low = model_rfm[col].quantile(0.01)
    high = model_rfm[col].quantile(0.99)
    model_rfm[col] = model_rfm[col].clip(low, high)

In [ ]:
# 7. STANDARDISATION

In [ ]:
features = ['Recency', 'Frequency', 'Monetary']
scaler = StandardScaler()
X = scaler.fit_transform(model_rfm[features])

In [ ]:
# 8. ELBOW METHOD

In [ ]:
k_values = range(2, 11)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

plt.figure(figsize=(9, 5))
plt.plot(list(k_values), inertias, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(list(k_values))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('output/elbow_method.png', dpi=150)
plt.show()

scores = pd.DataFrame({'K': list(k_values), 'SilhouetteScore': silhouettes})
print('\nSilhouette scores:')
print(scores)
scores.to_csv('output/silhouette_scores.csv', index=False)

# The elbow for this type of RFM dataset is commonly around 4–5.
# Use the highest silhouette score as a reproducible supporting criterion.
optimal_k = int(scores.loc[scores['SilhouetteScore'].idxmax(), 'K'])
print('Selected K:', optimal_k)

In [ ]:
# 9. K-MEANS CLUSTERING

In [ ]:
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(X)

In [ ]:
# 10. CLUSTER PROFILE

In [ ]:
cluster_profile = rfm.groupby('Cluster').agg(
    Customers=('CustomerID', 'count'),
    AvgRecency=('Recency', 'mean'),
    AvgFrequency=('Frequency', 'mean'),
    AvgMonetary=('Monetary', 'mean'),
    TotalRevenue=('Monetary', 'sum')
).round(2)

# Higher frequency/monetary and lower recency indicate stronger customers.
cluster_profile['ValueScore'] = (
    cluster_profile['AvgFrequency'].rank(pct=True)
    + cluster_profile['AvgMonetary'].rank(pct=True)
    + (1 - cluster_profile['AvgRecency'].rank(pct=True))
)
cluster_profile = cluster_profile.sort_values('ValueScore', ascending=False)

# Business-friendly labels based on relative RFM strength.
labels = ['Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk',
          'Need Attention', 'Lost / Low Value', 'Regular Customers',
          'Occasional Customers', 'Other', 'Other']
cluster_to_segment = {cluster: labels[i] for i, cluster in enumerate(cluster_profile.index)}
rfm['Segment'] = rfm['Cluster'].map(cluster_to_segment)

print('\nCluster profile:')
print(cluster_profile)

cluster_profile.to_csv('output/cluster_profile.csv')

In [ ]:
# 11. BAR CHART: CUSTOMERS PER CLUSTER

In [ ]:
counts = rfm.groupby(['Cluster', 'Segment']).size().reset_index(name='Customers')
plt.figure(figsize=(10, 5))
sns.barplot(data=counts, x='Segment', y='Customers')
plt.title('Number of Customers per Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('output/customers_per_segment.png', dpi=150)
plt.show()

In [ ]:
# 12. SCATTER PLOTS: TWO FEATURE COMBINATIONS

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Segment', s=70)
plt.title('Customer Segments: Frequency vs Monetary')
plt.xlabel('Purchase Frequency')
plt.ylabel('Monetary Value')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('output/frequency_vs_monetary.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Segment', s=70)
plt.title('Customer Segments: Recency vs Monetary')
plt.xlabel('Recency (days)')
plt.ylabel('Monetary Value')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('output/recency_vs_monetary.png', dpi=150)
plt.show()

In [ ]:
# 13. EXPORT RESULTS

In [ ]:
rfm.to_csv('output/rfm_analysis.csv', index=False)
rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'Segment']].to_csv(
    'output/customer_segments.csv', index=False
)

In [ ]:
# 14. MARKETING INSIGHTS

In [ ]:
print('\n--- Marketing Recommendations ---')
for segment in rfm['Segment'].dropna().unique():
    print(f'\n{segment}:')
    if segment == 'Champions':
        print('Reward with loyalty benefits, early access, premium offers and referral programs.')
    elif segment == 'Loyal Customers':
        print('Use cross-selling, personalized recommendations and loyalty rewards.')
    elif segment == 'Potential Loyalists':
        print('Encourage repeat purchases with bundles, reminders and limited-time offers.')
    elif segment == 'At Risk':
        print('Run re-engagement campaigns, personalized discounts and win-back messages.')
    elif segment == 'Need Attention':
        print('Use targeted promotions to increase purchase frequency and basket size.')
    elif segment == 'Lost / Low Value':
        print('Use low-cost win-back campaigns; avoid excessive marketing spend.')
    else:
        print('Use personalized offers and monitor behaviour for movement into higher-value segments.')

print('\nProject completed. Check the output/ folder for CSV files and charts.')